In [2]:
%%capture
!pip install keras-nlp

In [3]:
import pandas as pd
import numpy as np

import tensorflow as tf
import keras_nlp

from sklearn.preprocessing import LabelEncoder
import warnings

from zipfile import ZipFile


warnings.filterwarnings("ignore")

In [4]:
path = "/content/archive.zip"

with ZipFile(path, 'r') as zip:
  zip.extractall()
  print('Done')

Done


In [5]:
path_data = "/content/Combined Data.csv"
data = pd.read_csv(path_data, index_col=0)
data.dropna(inplace=True)
data.head()

,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety


In [6]:
training_size = 0.8
SEED = 42
BUFFER_SIZE = 10000
BATCH_SIZE = 16
n = 20000
AUTOTUNE = tf.data.AUTOTUNE

In [7]:
data = data.sample(n=n, random_state=SEED)

In [8]:
label_encoder = LabelEncoder()

statement = data["statement"].values
status  = label_encoder.fit_transform(data["status"])

In [9]:
classes = label_encoder.classes_

# Display the correspondence between indexes and status
for index, class_name in enumerate(classes):
    print(f"Index: {index}, Status: {class_name}")

Index: 0, Status: Anxiety
Index: 1, Status: Bipolar
Index: 2, Status: Depression
Index: 3, Status: Normal
Index: 4, Status: Personality disorder
Index: 5, Status: Stress
Index: 6, Status: Suicidal


In [10]:
dataset = tf.data.Dataset.from_tensor_slices((statement, status))

training_size = int(training_size * len(statement))

train_ds = dataset.take(training_size)
test_ds = dataset.skip(training_size)

train_dataset = train_ds.shuffle(buffer_size=BUFFER_SIZE).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
test_dataset  = test_ds.batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

train_dataset.element_spec

(TensorSpec(shape=(None,), dtype=tf.string, name=None),
 TensorSpec(shape=(None,), dtype=tf.int64, name=None))

In [11]:
for example, label in train_dataset.take(1):
  print('text: ', example.numpy())
  print('label: ', label.numpy())

text:  [b'she wa the only thing making me feel guilt or sadness at the thought but now knowing she won t care i think i m ready'
 b'I had terrible depression after my grandmother died in 2018. I was in a free fall and could not get myself out. I was pressured by my (My parents both died and i live with these idiots) uncle and aunt to perform in school, which i did not at the time. I was in a tiny kind of high school relationship with a girl and it was horrible, i was her baggage off-loader. Fast forward 1 year and I am doing much better, grades skyrocket started playing sports again and doing choir. Life started looking up for a bunch of years until now And its bad.This year i met a girl, and i really do love her. My issues just fly out of the door when I am with her. But when i get home, i want to die. I started studying BSc computer science. Its tough, but its my passion. I am losing my emotions, i cannot laugh I cannot cry, I am only down the whole time. Me and my uncle had an argum

In [12]:
num_classes = len(np.unique(status))

In [13]:
"""
BERT:

"bert_base_en"
"bert_large_en"

XLNet:

"xlnet_base_en"
"xlnet_large_en"

RoBERTa:

"roberta_base_en"
"roberta_large_en"

DistilBERT:

"distilbert_base_en"

ELECTRA:

"electra_base_en"
"electra_large_en"

ALBERT:

"albert_base_en"
"albert_large_en"

GPT-2:

"gpt2_base_en"
"gpt2_large_en"
"""

'\nBERT:\n\n"bert_base_en"\n"bert_large_en"\n\nXLNet:\n\n"xlnet_base_en"\n"xlnet_large_en"\n\nRoBERTa:\n\n"roberta_base_en"\n"roberta_large_en"\n\nDistilBERT:\n\n"distilbert_base_en"\n\nELECTRA:\n\n"electra_base_en"\n"electra_large_en"\n\nALBERT:\n\n"albert_base_en"\n"albert_large_en"\n\nGPT-2:\n\n"gpt2_base_en"\n"gpt2_large_en"\n'

In [15]:
classifier = keras_nlp.models.Classifier.from_preset(
    "roberta_base_en",
    num_classes=num_classes,
)

100%|██████████| 141/141 [00:00<00:00, 354kB/s]


100%|██████████| 498/498 [00:00<00:00, 384kB/s]


100%|██████████| 474M/474M [00:09<00:00, 51.3MB/s]


100%|██████████| 463/463 [00:00<00:00, 465kB/s]


100%|██████████| 0.99M/0.99M [00:00<00:00, 3.02MB/s]


100%|██████████| 446k/446k [00:00<00:00, 1.56MB/s]


In [16]:
classifier.fit(train_dataset, validation_data=test_dataset)

500/500 ━━━━━━━━━━━━━━━━━━━━ 912s 2s/step - loss: 0.9667 - sparse_categorical_accuracy: 0.6230 - val_loss: 0.5453 - val_sparse_categorical_accuracy: 0.7900


In [42]:
predict = classifier.predict(["IYS ASTRONOMYYYY"])
predicted_class = predict.argmax()

print(predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
3


In [ ]:
"""
Índice: 0, Status: Anxiety
Índice: 1, Status: Bipolar
Índice: 2, Status: Depression
Índice: 3, Status: Normal
Índice: 4, Status: Personality disorder
Índice: 5, Status: Stress
Índice: 6, Status: Suicidal
"""